[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_18/MFM_ch18_covar_tests/MFM_ch18_covar_tests.ipynb)

# MFM_ch18_covar_tests

Inference for systemic-risk measures of 11 US and European banks: joint moving-block bootstrap (999 draws) tests of Delta-CoVaR = 0, of equal tail and median slopes and of equal contributions (Holm correction, joint Wald test), the Euler identity ES = sum of MES when conditioning on the portfolio itself, the Girardi-Ergun CoVaR at 5% and an out-of-sample backtest of its joint hits.

*Modelling Financial Markets (MFM), Chapter 18: Systemic Risk and Networks. Author: Daniel Traian Pele.*

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats
from scipy.stats import spearmanr
import statsmodels.api as sm
from statsmodels.regression.quantile_regression import QuantReg
from sklearn.linear_model import QuantileRegressor, ElasticNetCV
from scipy import optimize
try:
    from arch import arch_model
except ImportError:                      # Colab: the arch package is not preinstalled
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'arch'])
    from arch import arch_model
import warnings
warnings.filterwarnings('ignore')

## Style and helpers

In [ ]:
# Chart style: transparent background, legend below the plot
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Course colours
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Teal     = '#17A2B8'
Navy     = '#1F2A44'   # reference lines (no grey in charts)
BandBlue = '#C5D2E8'   # light MainBlue tint for confidence / reference bands
Gray, LightGray = Navy, BandBlue   # legacy names kept for importing scripts
REGION_COL = {'US': MainBlue, 'EU': IDAred, 'RO': Forest}
REGION_LAB = {'US': 'US banks', 'EU': 'European banks', 'RO': 'Romanian banks (BVB)'}
EVENTS = [('2011-08-08', 'Euro crisis'), ('2016-06-24', 'Brexit vote'), ('2018-12-19', 'RO bank tax'),
          ('2020-03-16', 'COVID-19'), ('2023-03-13', 'SVB'), ('2025-04-07', 'Tariffs')]

HERE = '.'
SEED = 42
B_BOOT = 999
RESULTS = {}


def save_fig(name):
    """Save the figure as transparent PDF and PNG in the current folder."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Place the legend outside the plot, bottom centre."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def jsonable(x):
    if isinstance(x, dict):
        return {str(k): jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [jsonable(v) for v in x]
    if isinstance(x, np.ndarray):
        return [jsonable(v) for v in x.tolist()]
    if isinstance(x, (np.floating, np.integer)):
        return x.item()
    if isinstance(x, pd.Timestamp):
        return str(x.date())
    return x


def mark_events(ax, ymax=None):
    """Vertical reference lines for events, with black labels."""
    for d, lab in EVENTS:
        t = pd.Timestamp(d)
        if ax.get_xlim()[0] <= mdates.date2num(t) <= ax.get_xlim()[1]:
            ax.axvline(t, color=Gray, lw=0.5, ls=':')
            ax.text(t, ax.get_ylim()[1] if ymax is None else ymax, lab, rotation=90, fontsize=6.5, color='black',
                    ha='right', va='top')

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
START = '2010-01-01'
END = '2026-09-18'

# bank -> (symbol, name, region)
BANKS = {
    'JPM':  ('JPM.US',    'JPMorgan Chase',       'US'),
    'BAC':  ('BAC.US',    'Bank of America',      'US'),
    'C':    ('C.US',      'Citigroup',            'US'),
    'GS':   ('GS.US',     'Goldman Sachs',        'US'),
    'MS':   ('MS.US',     'Morgan Stanley',       'US'),
    'WFC':  ('WFC.US',    'Wells Fargo',          'US'),
    'DBK':  ('DBK.XETRA', 'Deutsche Bank',        'EU'),
    'BNP':  ('BNP.PA',    'BNP Paribas',          'EU'),
    'SAN':  ('SAN.MC',    'Santander',            'EU'),
    'INGA': ('INGA.AS',   'ING',                  'EU'),
    'HSBA': ('HSBA.LSE',  'HSBC',                 'EU'),
    'TLV':  ('TLV.RO',    'Banca Transilvania',   'RO'),
    'BRD':  ('BRD.RO',    'BRD-Groupe SG',        'RO'),
}
US = [k for k, v in BANKS.items() if v[2] == 'US']
EU = [k for k, v in BANKS.items() if v[2] == 'EU']
RO = [k for k, v in BANKS.items() if v[2] == 'RO']
ALL = US + EU + RO
NAMES = {k: v[1] for k, v in BANKS.items()}
INDICES = {'SPX': ('GSPC.INDX', 'S&P 500'), 'SX5E': ('STOXX50E.INDX', 'Euro Stoxx 50'),
           'BET': ('BET', 'BET'), 'VIX': ('VIX.INDX', 'VIX')}
REGION_INDEX = {'US': 'SPX', 'EU': 'SX5E', 'RO': 'BET'}


def read_market(symbol):
    """Read the daily price series of one symbol from the course data."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def bank_frame(key, start=START, end=END):
    """Daily prices and volume of one bank, with the adjusted price and the corrections above."""
    symbol, _, region = BANKS[key]
    d = read_market(symbol).loc[start:end].copy()
    d = d[(d.index.dayofweek < 5) & (d['adjusted_close'] > 0)]
    if region == 'RO':
        d = d[d['volume'] > 0]                                   # days without trades
    if key == 'TLV' and pd.Timestamp('2016-05-30') in d.index:
        # bonus-share adjustment: the post-ex-date factor also applies on 30 May 2016
        f = d.loc['2016-05-31', 'adjusted_close'] / d.loc['2016-05-31', 'close']
        d.loc['2016-05-30', 'adjusted_close'] = d.loc['2016-05-30', 'close'] * f
    return d


def bank_price(key, start=START, end=END):
    return bank_frame(key, start, end)['adjusted_close'].rename(key)


def index_close(key, start=START, end=END):
    s = read_market(INDICES[key][0])['close'].loc[start:end]
    s = s[(s.index.dayofweek < 5) & (s > 0)].dropna()
    if key != 'VIX':
        s = s[s.diff() != 0]                                     # holidays filled with the previous price
    return s.rename(key)


def prices(keys, start=START, end=END):
    """Daily prices of banks and/or indices, on common trading days only."""
    cols = [bank_price(k, start, end) if k in BANKS else index_close(k, start, end) for k in keys]
    return pd.concat(cols, axis=1, join='inner').dropna()


def joint_returns(keys, start=START, end=END):
    """Daily log returns in %: prices are aligned on common trading days first, then returns are computed."""
    return 100 * np.log(prices(keys, start, end)).diff().dropna()


def bank_range_vol(keys, start=START, end=END):
    """Daily Parkinson volatility (in %, annualised), on the days common to all banks."""
    out = []
    for k in keys:
        d = bank_frame(k, start, end)
        rng = np.log(d['high'] / d['low']).where(d['high'] > d['low'])
        s2 = rng ** 2 / (4 * np.log(2))
        floor = s2.where(s2 > 0).cummin().ffill()                  # smallest positive value observed UP TO day t
        s2 = s2.fillna(floor)                                     # days with high = low: no look-ahead bias
        out.append((100 * np.sqrt(252 * s2)).rename(k))
    return pd.concat(out, axis=1, join='inner').dropna()


def system_return(R, members):
    """System return: equal-weighted portfolio of the banks (log returns in %, daily approximation)."""
    return R[members].mean(axis=1)


# daily log returns in % (common days) and log Parkinson volatility
R = joint_returns(ALL + ['SPX', 'SX5E', 'BET', 'VIX'])
V = np.log(bank_range_vol(ALL))
print(R.index[0].date(), '->', R.index[-1].date(), len(R), 'common days')


def system_ex(k, members):
    """Return of the regional system without bank k (equal-weighted portfolio)."""
    return R[[j for j in members if j != k]].mean(axis=1)


def region_of(k):
    return BANKS[k][2]

## Systemic risk measures

In [ ]:
def mes(r_i, r_m, alpha=0.05):
    """MES (%): average loss of the bank on the days when the market is in its alpha-probability tail."""
    q = np.quantile(r_m, alpha)
    return -np.mean(np.asarray(r_i)[np.asarray(r_m) <= q])


def mes_threshold(r_i, r_m, c=-2.0):
    """MES (%) on the days when the market falls by more than |c|% (the definition used for LRMES)."""
    return -np.mean(np.asarray(r_i)[np.asarray(r_m) < c])


def lrmes(mes2_pct):
    """Long-run loss in a crisis (market -40% over six months): LRMES = 1 - exp(-18 MES), MES as a fraction."""
    return 1 - np.exp(-18 * mes2_pct / 100)


def srisk_ratio(lrm, leverage, k=0.08):
    """SRISK / market equity W: k(L - 1) - (1 - k)(1 - LRMES), with L = (D + W)/W."""
    return k * (leverage - 1) - (1 - k) * (1 - lrm)


def breakeven_leverage(lrm, k=0.08):
    """Leverage L* above which SRISK > 0: L* = 1 + (1 - k)(1 - LRMES)/k."""
    return 1 + (1 - k) * (1 - lrm) / k


def qreg(y, X, q):
    """Quantile regression (Koenker-Bassett); returns the coefficients [intercept, slopes]."""
    return QuantReg(np.asarray(y), sm.add_constant(np.asarray(X), has_constant='add')).fit(q=q, max_iter=5000).params


def covar_static(x_sys, x_i, alpha=0.01):
    """Unconditional CoVaR and Delta-CoVaR (%, positive = losses).

    Quantile regression at level alpha: X_sys = a + b X_i;
    CoVaR_alpha = -(a + b q_alpha(X_i)); Delta-CoVaR = b (q_50(X_i) - q_alpha(X_i))."""
    x_sys, x_i = np.asarray(x_sys), np.asarray(x_i)
    a, b = qreg(x_sys, x_i, alpha)
    qa, qm = np.quantile(x_i, alpha), np.quantile(x_i, 0.5)
    return {'a': a, 'b': b, 'var_i': -qa, 'covar': -(a + b * qa), 'covar_med': -(a + b * qm),
            'dcovar': b * (qm - qa), 'var_sys': -np.quantile(x_sys, alpha)}


def block_bootstrap_idx(n, block=20, rng=None):
    """Indices for the moving-block bootstrap (Künsch), fixed-length blocks."""
    rng = rng or np.random.default_rng(0)
    starts = rng.integers(0, n - block + 1, size=int(np.ceil(n / block)))
    return (starts[:, None] + np.arange(block)[None, :]).ravel()[:n]


def covar_boot(x_sys, x_i, alpha=0.01, B=200, block=20, seed=0):
    """95% percentile interval for Delta-CoVaR by moving-block bootstrap."""
    x_sys, x_i = np.asarray(x_sys), np.asarray(x_i)
    rng = np.random.default_rng(seed)
    d = []
    for _ in range(B):
        idx = block_bootstrap_idx(len(x_i), block, rng)
        d.append(covar_static(x_sys[idx], x_i[idx], alpha)['dcovar'])
    return np.percentile(d, [2.5, 97.5])


def covar_dynamic(x_sys, x_i, M, alpha=0.01):
    """Time-varying Delta-CoVaR with state variables M_{t-1} (Adrian & Brunnermeier, 2016).

    X_i,t = a + c'M_{t-1} at levels alpha and 50%  ->  VaR_i,t;
    X_sys,t = a + b X_i,t + c'M_{t-1} at level alpha  ->  Delta-CoVaR_t = b (q50_i,t - qalpha_i,t)."""
    ci_a = qreg(x_i, M, alpha)
    ci_m = qreg(x_i, M, 0.5)
    cs = qreg(x_sys, np.column_stack([x_i, M]), alpha)
    Mc = sm.add_constant(np.asarray(M), has_constant='add')
    qa, qm = Mc @ ci_a, Mc @ ci_m
    return pd.DataFrame({'var_i': -qa, 'dcovar': cs[1] * (qm - qa)}, index=getattr(x_i, 'index', None)), cs[1]


def var_fit(Y, p):
    """VAR(p) estimated by OLS; returns the coefficients A_1..A_p (k x k) and the residual covariance matrix."""
    Y = np.asarray(Y, float)
    T, k = Y.shape
    X = np.hstack([np.ones((T - p, 1))] + [Y[p - l - 1:T - l - 1] for l in range(p)])
    Yt = Y[p:]
    B = np.linalg.lstsq(X, Yt, rcond=None)[0]
    E = Yt - X @ B
    S = E.T @ E / (len(Yt) - X.shape[1])
    A = [B[1 + l * k:1 + (l + 1) * k].T for l in range(p)]
    return A, S


def var_bic(Y, pmax=5):
    """VAR order chosen by the Bayesian information criterion (BIC, Schwarz), with the maximum-likelihood residual covariance matrix."""
    Y = np.asarray(Y, float)
    T, k = Y.shape
    best = None
    n = T - pmax
    for p in range(1, pmax + 1):
        A, S = var_fit(Y[pmax - p:], p)
        S_ml = S * (n - 1 - p * k) / n                        # maximum-likelihood covariance E'E / n
        bic = np.log(np.linalg.det(S_ml)) + np.log(n) * p * k * k / n
        if best is None or bic < best[1]:
            best = (p, bic)
    return best[0]


def ma_coefs(A, H):
    """Moving-average coefficients Phi_0..Phi_{H-1}."""
    k = A[0].shape[0]
    Phi = [np.eye(k)]
    for h in range(1, H):
        Phi.append(sum(A[l] @ Phi[h - l - 1] for l in range(min(h, len(A)))))
    return Phi


def gfevd(A, S, H=10):
    """Generalized forecast error variance decomposition (Pesaran-Shin), rows normalised."""
    Phi = ma_coefs(A, H)
    k = S.shape[0]
    num = np.zeros((k, k))
    den = np.zeros(k)
    for P in Phi:
        PS = P @ S
        num += PS ** 2
        den += np.diag(P @ S @ P.T)
    theta = num / np.diag(S)[None, :] / den[:, None]
    return theta / theta.sum(axis=1, keepdims=True)


def spillover_table(theta, names):
    """Connectedness table: contributions (%), FROM, TO, NET and the total index."""
    k = theta.shape[0]
    D = 100 * theta
    off = D - np.diag(np.diag(D))
    frm, to = off.sum(axis=1), off.sum(axis=0)
    tab = pd.DataFrame(D, index=names, columns=names)
    return {'table': tab, 'from': pd.Series(frm, index=names), 'to': pd.Series(to, index=names),
            'net': pd.Series(to - frm, index=names), 'total': off.sum() / k,
            'pairwise_net': pd.DataFrame(off.T - off, index=names, columns=names)}


def rolling_total(Y, window=250, step=5, p=2, H=10):
    """Total connectedness index on rolling windows."""
    Y = pd.DataFrame(Y)
    out, frm_to = {}, {}
    for end in range(window, len(Y) + 1, step):
        w = Y.iloc[end - window:end]
        A, S = var_fit(w.values, p)
        st = spillover_table(gfevd(A, S, H), list(Y.columns))
        out[Y.index[end - 1]] = st['total']
        frm_to[Y.index[end - 1]] = st['net']
    return pd.Series(out), pd.DataFrame(frm_to).T


def check_loss(u, tau):
    return u * (tau - (u < 0))


def lasso_qr_gacv(y, X, tau=0.05, lambdas=None):
    """L1-penalized quantile regression; lambda chosen by the generalized approximate cross-validation criterion GACV = sum rho_tau(res) / (n - df)."""
    y, X = np.asarray(y, float), np.asarray(X, float)
    n = len(y)
    lambdas = np.logspace(-4, -0.5, 15) if lambdas is None else lambdas
    rows = []
    for lam in lambdas:
        m = QuantileRegressor(quantile=tau, alpha=lam, solver='highs').fit(X, y)
        res = y - m.predict(X)
        df = int(np.sum(np.abs(m.coef_) > 1e-8))
        g = check_loss(res, tau).sum() / max(n - df, 1)
        rows.append((lam, g, df, m.coef_.copy()))
    i = int(np.argmin([r[1] for r in rows]))
    return {'lambda': rows[i][0], 'gacv': [r[1] for r in rows], 'df': [r[2] for r in rows],
            'lambdas': list(lambdas), 'coef': rows[i][3], 'df_sel': rows[i][2]}

## Helper functions

In [ ]:
B = 999

## Analysis

In [ ]:
def holm(p):
    """Holm correction (adjusted p-values) for a family of tests."""
    p = np.asarray(p, float)
    o = np.argsort(p)
    m = len(p)
    adj = np.maximum.accumulate((m - np.arange(m)) * p[o])
    out = np.empty(m)
    out[o] = np.minimum(adj, 1)
    return out


def covar_tests(alpha=0.01, B=B, block=20, seed=SEED):
    """The same bootstrap draws for all banks -> joint distribution of Delta-CoVaR and of the slopes."""
    banks = US + EU
    sysr = {k: system_ex(k, US if k in US else EU).values for k in banks}
    x = {k: R[k].values for k in banks}
    n = len(R)
    point = {k: covar_static(sysr[k], x[k], alpha) for k in banks}
    b50 = {k: qreg(sysr[k], x[k], 0.5)[1] for k in banks}
    rng = np.random.default_rng(seed)
    D = np.zeros((B, len(banks)))
    DB = np.zeros((B, len(banks)))
    for r in range(B):
        idx = block_bootstrap_idx(n, block, rng)
        for j, k in enumerate(banks):
            c = covar_static(sysr[k][idx], x[k][idx], alpha)
            D[r, j] = c['dcovar']
            DB[r, j] = c['b'] - qreg(sysr[k][idx], x[k][idx], 0.5)[1]
    res = {'alpha': alpha, 'n': n, 'n_tail': n * alpha, 'banks': {}}
    for j, k in enumerate(banks):
        fit = QuantReg(sysr[k], sm.add_constant(x[k])).fit(q=alpha, max_iter=5000)     # kernel standard error (sandwich)
        dpt = point[k]['b'] - b50[k]
        se_db = DB[:, j].std(ddof=1)
        res['banks'][k] = {'dcovar': point[k]['dcovar'], 'b': point[k]['b'], 'b50': b50[k], 'se_b_kernel': fit.bse[1],
                           'se_b_boot': None, 'lo': np.percentile(D[:, j], 2.5), 'hi': np.percentile(D[:, j], 97.5),
                           'se_dcovar_boot': D[:, j].std(ddof=1), 'db': dpt, 'se_db': se_db,
                           'p_db': 2 * stats.norm.sf(abs(dpt) / se_db)}
    # equal contributions by pair, within each region: t = difference / bootstrap standard deviation of the difference
    pairs = []
    for mem in (US, EU):
        for a in range(len(mem)):
            for b_ in range(a + 1, len(mem)):
                i, j = banks.index(mem[a]), banks.index(mem[b_])
                d = point[mem[a]]['dcovar'] - point[mem[b_]]['dcovar']
                se = (D[:, i] - D[:, j]).std(ddof=1)
                pairs.append((mem[a], mem[b_], d, se, 2 * stats.norm.sf(abs(d) / se),
                              # overlap of the marginal intervals
                              not (res['banks'][mem[a]]['hi'] < res['banks'][mem[b_]]['lo'] or
                                   res['banks'][mem[b_]]['hi'] < res['banks'][mem[a]]['lo'])))
    p = np.array([q[4] for q in pairs])
    ph = holm(p)
    res['pairs'] = [{'a': q[0], 'b': q[1], 'd': q[2], 'se': q[3], 'p': q[4], 'p_holm': ph[i], 'overlap': q[5]}
                    for i, q in enumerate(pairs)]
    res['n_pairs'] = len(pairs)
    res['n_sig_raw'] = int((p < 0.05).sum())
    res['n_sig_holm'] = int((ph < 0.05).sum())
    res['n_sig_overlap'] = int(sum(1 for i, q in enumerate(pairs) if q[5] and p[i] < 0.05))
    res['n_db_sig'] = int(sum(v['p_db'] < 0.05 for v in res['banks'].values()))
    res['n_db_sig_holm'] = int((holm([v['p_db'] for v in res['banks'].values()]) < 0.05).sum())
    # joint Wald test: all Delta-CoVaR equal within a region (bootstrap covariance of the contrasts)
    for reg, mem in (('US', US), ('EU', EU)):
        ids = [banks.index(k) for k in mem]
        C = np.zeros((len(ids) - 1, len(banks)))
        for r_, i in enumerate(ids[1:]):
            C[r_, ids[0]], C[r_, i] = 1, -1
        dv = C @ np.array([point[k]['dcovar'] for k in banks])
        S = np.cov((D @ C.T).T)
        w = float(dv @ np.linalg.solve(S, dv))
        res[f'wald_{reg}'] = {'stat': w, 'df': len(ids) - 1, 'p': stats.chi2.sf(w, len(ids) - 1)}
    return res


def euler_check(alpha=0.05):
    X = R[US]
    port = X.mean(axis=1)
    q = np.quantile(port, alpha)
    es_p = -port[port <= q].mean()
    mes_p = {k: -X[k][port <= q].mean() for k in US}
    m = R['SPX']
    mes_m = {k: mes(X[k].values, m.values, alpha) for k in US}
    return {'es_port': es_p, 'sum_mes_port': np.mean(list(mes_p.values())), 'sum_mes_spx': np.mean(list(mes_m.values())),
            'mes_port': mes_p, 'mes_spx': mes_m, 'n_tail': int((port <= q).sum())}


def ge_covar(alpha=0.05):
    out = {}
    for k in US + EU:
        xs = system_ex(k, US if k in US else EU).values
        xi = R[k].values
        qa, qm = np.quantile(xi, alpha), np.quantile(xi, 0.5)
        dist = xs[xi <= qa]
        med = xs[np.abs(xi - xi.mean()) <= xi.std()]                       # benchmark state: a one-standard-deviation event
        out[k] = {'ge_covar': -np.quantile(dist, alpha), 'ge_covar_med': -np.quantile(med, alpha),
                  'ab_covar': covar_static(xs, xi, alpha)['covar'], 'n_cond': int(len(dist))}
        out[k]['ge_dcovar'] = out[k]['ge_covar'] - out[k]['ge_covar_med']
    return out


def kupiec(x, n, p):
    """Kupiec test (unconditional coverage): likelihood-ratio statistic LR ~ chi2(1) under H0: hit rate = p."""
    ph = x / n
    ll0 = x * np.log(p) + (n - x) * np.log(1 - p)
    ll1 = (x * np.log(ph) if x > 0 else 0) + ((n - x) * np.log(1 - ph) if x < n else 0)
    lr = -2 * (ll0 - ll1)
    return lr, stats.chi2.sf(lr, 1)


def backtest_ge(alpha=0.05, start=1000):
    """Out-of-sample backtest of the Girardi-Ergun CoVaR (historical, expanding window):
    joint hit 1{X_sys <= -CoVaR_t, X_i <= -VaR_i,t}, with probability alpha^2 under H0."""
    out = {}
    for k in US + EU:
        xs = system_ex(k, US if k in US else EU).values
        xi = R[k].values
        hits, vh = [], []
        for t in range(start, len(xi)):
            a, s = xi[:t], xs[:t]
            q = np.quantile(a, alpha)
            cov = -np.quantile(s[a <= q], alpha)
            vh.append(xi[t] <= q)
            hits.append((xi[t] <= q) and (xs[t] <= -cov))
        n, x = len(hits), int(np.sum(hits))
        lr, p = kupiec(x, n, alpha ** 2)
        lrv, pv = kupiec(int(np.sum(vh)), n, alpha)
        out[k] = {'n': n, 'hits': x, 'expected': n * alpha ** 2, 'lr': lr, 'p': p,
                  'var_hits': int(np.sum(vh)), 'var_p': pv}
    return out

## Run

In [ ]:
ct = covar_tests(0.01)
print({k: ct[k] for k in ['n_pairs', 'n_sig_raw', 'n_sig_holm', 'wald_US', 'wald_EU']})
print(euler_check()['es_port'], euler_check()['sum_mes_port'])
print(pd.DataFrame(ge_covar()).T.round(2))
pd.DataFrame(backtest_ge(0.05)).T.round(3)